In [77]:
import os
from pathlib import Path
from PIL import Image
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import time

In [78]:
df = pd.read_excel("data_pre_1.xlsx")

In [79]:
df.head(5)

,category,id,title,url,path
0,Điện Thoại - Máy Tính Bảng,278237932,Điện thoại Xiaomi Redmi 14C - Hàng chính hãng,https://salt.tikicdn.com/cache/280x280/ts/prod...,images/phone/278237932.png
1,Điện Thoại - Máy Tính Bảng,278098703,Điện thoại POCO C75 (8GB/256GB) - Hàng Chính Hãng,https://salt.tikicdn.com/cache/280x280/ts/prod...,images/phone/278098703.png
2,Điện Thoại - Máy Tính Bảng,278000720,Điện thoại HONOR X5b Plus 4GB/128GB - Hàng chí...,https://salt.tikicdn.com/cache/280x280/ts/prod...,images/phone/278000720.jpg
3,Điện Thoại - Máy Tính Bảng,277930407,Điện thoại Tecno Spark GO 1 (3GB/64GB) - Hàng ...,https://salt.tikicdn.com/cache/280x280/ts/prod...,images/phone/277930407.png
4,Điện Thoại - Máy Tính Bảng,277777809,"Điện thoại Samsung Galaxy A26 5G (8/128GB), Mặ...",https://salt.tikicdn.com/cache/280x280/ts/prod...,images/phone/277777809.jpg


In [80]:
df.isnull().sum().sum()

np.int64(0)

In [81]:
df.isna().sum().sum()

np.int64(0)

In [82]:
if "path" not in df.columns:
    raise ValueError("File không có cột 'path'!")

# Lấy phần mở rộng (đuôi file) của từng ảnh
df["ext"] = df["path"].dropna().astype(str).apply(lambda x: Path(x).suffix.lower())

# Thống kê các định dạng
ext_counts = df["ext"].value_counts()

print("📊 Các định dạng ảnh hiện có:")
print(ext_counts)

📊 Các định dạng ảnh hiện có:
ext
.jpg     358
.png     137
.jpeg      5
Name: count, dtype: int64


In [83]:
crawl_data_dir = "../crawl_data/images"
output_dir = "images_png"  # Thư mục mới để lưu ảnh PNG
categories = ["camera", "laptop", "phone", "speaker", "tv"]

In [84]:
# Cấu hình resize
TARGET_SIZE = (224, 224)  # Kích thước mong muốn
RESIZE_ENABLED = True     # Bật/tắt resize
RESIZE_METHOD = 'padding' # 'padding', 'crop', hoặc 'stretch'

# Tạo thư mục output nếu chưa tồn tại
os.makedirs(output_dir, exist_ok=True)

In [85]:
# Hàm resize ảnh với padding
def resize_image(img, target_size):
    img_ratio = img.width / img.height
    target_ratio = target_size[0] / target_size[1]
    
    if img_ratio > target_ratio:
        new_height = target_size[1]
        new_width = int(new_height * img_ratio)
    else:
        new_width = target_size[0]
        new_height = int(new_width / img_ratio)
    
    img_resized = img.resize((new_width, new_height), Image.LANCZOS)
    new_img = Image.new('RGB', target_size, (255, 255, 255))
    
    x_offset = (target_size[0] - new_width) // 2
    y_offset = (target_size[1] - new_height) // 2
    
    new_img.paste(img_resized, (x_offset, y_offset))
    return new_img

# Hàm resize ảnh với crop
def resize_image_crop(img, target_size):
    img_ratio = img.width / img.height
    target_ratio = target_size[0] / target_size[1]
    
    if img_ratio > target_ratio:
        new_height = target_size[1]
        new_width = int(new_height * img_ratio)
        img_resized = img.resize((new_width, new_height), Image.LANCZOS)
        left = (new_width - target_size[0]) // 2
        right = left + target_size[0]
        img_resized = img_resized.crop((left, 0, right, target_size[1]))
    else:
        new_width = target_size[0]
        new_height = int(new_width / img_ratio)
        img_resized = img.resize((new_width, new_height), Image.LANCZOS)
        top = (new_height - target_size[1]) // 2
        bottom = top + target_size[1]
        img_resized = img_resized.crop((0, top, target_size[0], bottom))
    
    return img_resized

In [86]:
# Hàm chuyển đổi ảnh sang PNG
def convert_image_to_png(image_path, output_path):
    try:
        if not os.path.exists(image_path):
            print(f"File không tồn tại: {image_path}")
            return False
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with Image.open(image_path) as img:
            # Chuyển đổi mode
            if img.mode in ('RGBA', 'LA'):
                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[-1])
                img = background
            elif img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Resize ảnh nếu được bật
            if RESIZE_ENABLED:
                original_size = f"{img.width}x{img.height}"
                if RESIZE_METHOD == 'padding':
                    img = resize_image(img, TARGET_SIZE)
                elif RESIZE_METHOD == 'crop':
                    img = resize_image_crop(img, TARGET_SIZE)
                else:
                    img = img.resize(TARGET_SIZE, Image.LANCZOS)
                new_size = f"{img.width}x{img.height}"
                size_info = f" [{original_size} -> {new_size}]"
            else:
                size_info = ""
            
            # Lưu ảnh
            img.save(output_path, 'PNG', optimize=True)
            print(f"Đã chuyển đổi: {os.path.basename(image_path)}{size_info}")
            return True
            
    except Exception as e:
        print(f"Lỗi khi chuyển đổi {image_path}: {str(e)}")
        return False

In [87]:
# Hàm xử lý category
def process_category(category):
    category_path = os.path.join(crawl_data_dir, category)
    if not os.path.exists(category_path):
        print(f"Thư mục {category_path} không tồn tại!")
        return 0, 0
    
    converted_count = 0
    total_count = 0
    
    category_output_dir = os.path.join(output_dir, category)
    os.makedirs(category_output_dir, exist_ok=True)
    
    for filename in os.listdir(category_path):
        if filename.lower().endswith(('.jpg', '.jpeg')):
            image_path = os.path.join(category_path, filename)
            filename_without_ext = os.path.splitext(filename)[0]
            output_filename = f"{filename_without_ext}.png"
            output_path = os.path.join(category_output_dir, output_filename)
            
            # Bỏ qua nếu file đã tồn tại
            if os.path.exists(output_path):
                continue
            
            total_count += 1
            if convert_image_to_png(image_path, output_path):
                converted_count += 1
    
    return converted_count, total_count

# Hàm xử lý song song
def process_all_categories_parallel():
    results = {}
    
    with ThreadPoolExecutor(max_workers=len(categories)) as executor:
        future_to_category = {executor.submit(process_category, category): category for category in categories}
        
        for future in future_to_category:
            category = future_to_category[future]
            try:
                results[category] = future.result()
            except Exception as e:
                print(f"Lỗi khi xử lý category {category}: {str(e)}")
                results[category] = (0, 0)
    
    return results

In [88]:
# Hàm cập nhật đường dẫn trong DataFrame
def update_dataframe_paths(df):
    def update_path(path):
        if path and isinstance(path, str):
            filename = os.path.basename(path)
            filename_without_ext = os.path.splitext(filename)[0]
            old_dir = os.path.dirname(path)
            category = os.path.basename(old_dir)
            return os.path.join("images_png", category, f"{filename_without_ext}.png")
        return path
    
    df['path'] = df['path'].apply(update_path)
    return df

In [89]:
# Thực hiện chuyển đổi
print("Bắt đầu chuyển đổi ảnh sang PNG...")
print("=" * 60)
print(f"Ảnh PNG sẽ được lưu vào: {output_dir}")
print(f"Resize: {'CÓ' if RESIZE_ENABLED else 'KHÔNG'}")
if RESIZE_ENABLED:
    print(f"Kích thước: {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
    print(f"Phương pháp: {RESIZE_METHOD}")
print("=" * 60)

start_time = time.time()
results = process_all_categories_parallel()
end_time = time.time()

# In kết quả
print("=" * 60)
print("KẾT QUẢ CHUYỂN ĐỔI:")
print("=" * 60)

total_converted = 0
total_images = 0

for category, (converted, total) in results.items():
    print(f"{category.upper():<10}: {converted}/{total} ảnh đã chuyển đổi")
    total_converted += converted
    total_images += total

print("=" * 60)
print(f"TỔNG CỘNG: {total_converted}/{total_images} ảnh đã chuyển đổi thành công")
print(f"Thời gian: {end_time - start_time:.2f} giây")


Bắt đầu chuyển đổi ảnh sang PNG...
Ảnh PNG sẽ được lưu vào: images_png
Resize: CÓ
Kích thước: 224x224
Phương pháp: padding
Đã chuyển đổi: 276566181.jpg [280x280 -> 224x224]
Đã chuyển đổi: 4713289.jpg [280x280 -> 224x224]
Đã chuyển đổi: 181452535.jpg [280x280 -> 224x224]
Đã chuyển đổi: 274734984.jpg [280x280 -> 224x224]
Đã chuyển đổi: 2013189.jpg [280x280 -> 224x224]
Đã chuyển đổi: 277356246.jpg [280x280 -> 224x224]
Đã chuyển đổi: 176058957.jpg [280x280 -> 224x224]
Đã chuyển đổi: 273309458.jpg [280x280 -> 224x224]
Đã chuyển đổi: 275510578.jpg [280x280 -> 224x224]
Đã chuyển đổi: 273959250.jpg [280x280 -> 224x224]
Đã chuyển đổi: 249033305.jpg [280x280 -> 224x224]
Đã chuyển đổi: 263987622.jpg [280x280 -> 224x224]
Đã chuyển đổi: 278454662.jpg [280x280 -> 224x224]
Đã chuyển đổi: 252586291.jpg [280x280 -> 224x224]
Đã chuyển đổi: 274943200.jpeg [280x280 -> 224x224]
Đã chuyển đổi: 64672917.jpg [280x280 -> 224x224]
Đã chuyển đổi: 275245456.jpg [280x280 -> 224x224]
Đã chuyển đổi: 277672516.jpg [2

In [90]:
df = update_dataframe_paths(df)

In [91]:
df.to_excel("data_pre_2.xlsx", index=False)